MySQL 是后端面试里最核心的模块之一。它不像 Redis 那样偏缓存和高并发场景，也不像 Go runtime 那样偏语言底层；MySQL 更偏“业务数据的最终一致性、事务、索引、锁、复制、SQL 优化和工程排障”。

Go 项目里谈 MySQL，不能只会写：

```go
db.QueryContext(ctx, "select ...")
```

真正面试要讲清楚的是：

第一，Go 怎么接入 MySQL。
包括 `database/sql`、驱动、连接池、`sql.DB` 的真实含义、`QueryContext`、`ExecContext`、`Tx`、`Rows.Close`、连接泄漏、超时控制。

第二，MySQL 事务和 MVCC。
包括 ACID、四种隔离级别、脏读、不可重复读、幻读、Read View、undo log、当前读、快照读、RC 和 RR 的区别。

第三，日志与执行流程。
包括 redo log、undo log、binlog、两阶段提交、查询语句执行流程、更新语句执行流程、崩溃恢复。

第四，索引原理和 SQL 优化。
包括 B+ 树、聚簇索引、二级索引、回表、覆盖索引、最左前缀、索引下推、索引失效、EXPLAIN、慢查询。

第五，锁机制。
包括表锁、行锁、间隙锁、Next-Key Lock、共享锁、排他锁、死锁、锁等待、如何避免死锁。

第六，主从复制和读写分离。
包括 binlog、relay log、IO 线程、SQL 线程、主从延迟、强制读主、读一致性、Go 服务如何封装读写库。

第七，Go ORM 与工程实践。
包括 GORM、sqlx、手写 SQL 的取舍、事务封装、连接池参数、慢 SQL 监控、OpenTelemetry、Prometheus、幂等和唯一约束。

面试里可以把 MySQL 总结成一句话：

MySQL 面试的核心不是背 SQL 语法，而是理解 InnoDB 如何通过日志保证持久性，通过 MVCC 提升并发读，通过锁控制并发写，通过 B+ 树减少 IO，通过 binlog 实现复制；在 Go 项目里，还要把这些能力落实到连接池、事务边界、context 超时、SQL 优化和线上排障上。

---

# 一、Go 项目中如何接入 MySQL

## 【文本块 2】Q：Go 项目里通常怎么连接 MySQL？

Go 里最基础的数据库访问包是标准库 `database/sql`。

但 `database/sql` 本身只是抽象层，不包含具体数据库驱动。连接 MySQL 时通常会引入驱动：

```go
github.com/go-sql-driver/mysql
```

常见组合是：

```go
database/sql + go-sql-driver/mysql
```

或者：

```go
sqlx + go-sql-driver/mysql
```

再或者使用 ORM：

```go
GORM + go-sql-driver/mysql
```

其中：

`database/sql` 是标准库，稳定、轻量、显式。
`sqlx` 是对 `database/sql` 的增强，支持结构体映射，仍然保留 SQL 可控性。
GORM 是 ORM，开发效率高，但复杂 SQL 和性能敏感路径要谨慎。

Go 面试里要特别强调：

`sql.DB` 不是一个数据库连接，而是一个并发安全的连接池句柄。

它内部维护多个真实 TCP 连接。业务代码里通常在应用启动时初始化一个全局 `*sql.DB`，然后复用它，而不是每次请求都 Open 一个新的 DB。

---

## 【代码块 1】database/sql 连接 MySQL

```go
package main

import (
	"context"
	"database/sql"
	"fmt"
	"time"

	_ "github.com/go-sql-driver/mysql"
)

func main() {
	dsn := "user:password@tcp(127.0.0.1:3306)/demo?charset=utf8mb4&parseTime=true&loc=Local"

	db, err := sql.Open("mysql", dsn)
	if err != nil {
		fmt.Println("open mysql failed:", err)
		return
	}
	defer db.Close()

	db.SetMaxOpenConns(50)
	db.SetMaxIdleConns(10)
	db.SetConnMaxLifetime(30 * time.Minute)
	db.SetConnMaxIdleTime(5 * time.Minute)

	ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
	defer cancel()

	if err := db.PingContext(ctx); err != nil {
		fmt.Println("ping mysql failed:", err)
		return
	}

	fmt.Println("mysql connected")
}
```

---

## 【文本块 3】代码解释

这里有几个关键点。

第一，`sql.Open` 不一定立刻建立连接。
它更多是初始化一个 `*sql.DB` 句柄。真正验证连接可用，需要 `PingContext`。

第二，必须配置连接池参数。
否则默认值可能不适合生产环境。

第三，所有数据库操作都应该带 context。
比如 `PingContext`、`QueryContext`、`ExecContext`。这样可以控制超时和取消，避免数据库慢查询拖死 Go 服务。

第四，`parseTime=true` 很常用。
如果需要把 MySQL 的 datetime/timestamp 扫描到 Go 的 `time.Time`，通常需要加这个参数。

---

## 【文本块 4】Q：sql.DB 是连接吗？线程安全吗？

`sql.DB` 不是单个连接，而是连接池。

它是 goroutine-safe 的，可以被多个 goroutine 并发使用。

`sql.DB` 内部会根据需要创建、复用、回收连接。你可以通过下面几个参数控制连接池：

```go
SetMaxOpenConns
SetMaxIdleConns
SetConnMaxLifetime
SetConnMaxIdleTime
```

面试回答：

`sql.DB` 是数据库连接池抽象，不是单条连接。它是并发安全的，通常整个应用复用一个实例。生产中需要配置最大打开连接数、最大空闲连接数、连接最大生命周期和最大空闲时间，避免连接过多压垮 MySQL，也避免连接长期不释放导致异常。

---

# 二、Go MySQL 连接池参数

## 【文本块 5】Q：MySQL 连接池参数怎么配置？

最核心的四个参数是：

第一，`SetMaxOpenConns`。
最大打开连接数，也就是最多同时有多少个数据库连接。这个值太小，会导致请求等待连接；太大，会压垮 MySQL。

第二，`SetMaxIdleConns`。
最大空闲连接数。空闲连接太少，会频繁新建连接；太多，会浪费数据库资源。

第三，`SetConnMaxLifetime`。
连接最大生命周期。超过这个时间，连接会被关闭重建。它可以避免长连接长期存活导致被 MySQL、代理层或负载均衡器断开。

第四，`SetConnMaxIdleTime`。
连接最大空闲时间。空闲太久的连接会被释放。

工程配置不是死背数字，要结合：

* Go 服务实例数
* MySQL 最大连接数
* 请求 QPS
* 慢查询耗时
* 事务时长
* 是否有读写分离
* 是否有连接代理，比如 ProxySQL、RDS Proxy

例如 MySQL 最大连接数是 2000，服务有 20 个实例，那单实例最大连接数不能随便设 200，否则理论上就可能打满数据库。

面试里可以这样回答：

连接池参数要根据 MySQL 总连接数、服务实例数和请求耗时估算。`MaxOpenConns` 控制上限，防止把 MySQL 打爆；`MaxIdleConns` 减少频繁建连；`ConnMaxLifetime` 避免长连接老化；所有 SQL 调用还要配合 context timeout，避免连接被慢查询长期占用。

---

## 【代码块 2】连接池推荐封装

```go
package db

import (
	"context"
	"database/sql"
	"fmt"
	"time"

	_ "github.com/go-sql-driver/mysql"
)

type Config struct {
	DSN             string
	MaxOpenConns    int
	MaxIdleConns    int
	ConnMaxLifetime time.Duration
	ConnMaxIdleTime time.Duration
}

func NewMySQL(cfg Config) (*sql.DB, error) {
	db, err := sql.Open("mysql", cfg.DSN)
	if err != nil {
		return nil, fmt.Errorf("open mysql: %w", err)
	}

	db.SetMaxOpenConns(cfg.MaxOpenConns)
	db.SetMaxIdleConns(cfg.MaxIdleConns)
	db.SetConnMaxLifetime(cfg.ConnMaxLifetime)
	db.SetConnMaxIdleTime(cfg.ConnMaxIdleTime)

	ctx, cancel := context.WithTimeout(context.Background(), 3*time.Second)
	defer cancel()

	if err := db.PingContext(ctx); err != nil {
		_ = db.Close()
		return nil, fmt.Errorf("ping mysql: %w", err)
	}

	return db, nil
}
```

---

## 【文本块 6】追问：连接池太小和太大会有什么问题？

连接池太小：

* 请求等待连接
* RT 抖动
* goroutine 堆积
* 上游超时
* CPU 看起来不高，但服务吞吐上不去

连接池太大：

* MySQL 连接数被打满
* 每个连接都占 MySQL 内存
* 并发查询过多导致 MySQL CPU、IO 飙升
* 锁冲突加剧
* 整体吞吐反而下降

所以连接池不是越大越好，它本质上也是一种限流和资源隔离手段。

---

# 三、Query、QueryRow、Exec

## 【文本块 7】Q：Query、QueryRow、Exec 有什么区别？

`QueryContext` 用于返回多行结果。

```go
rows, err := db.QueryContext(ctx, sql, args...)
```

适合 SELECT 多行。

`QueryRowContext` 用于返回单行结果。

```go
err := db.QueryRowContext(ctx, sql, args...).Scan(...)
```

适合根据主键查一行、count 查询等。

`ExecContext` 用于不返回结果集的 SQL。

```go
result, err := db.ExecContext(ctx, sql, args...)
```

适合 INSERT、UPDATE、DELETE、DDL。

面试注意点：

`QueryContext` 返回的 `rows` 必须 Close。
否则连接不会及时归还连接池，最终可能导致连接池耗尽。

---

## 【代码块 3】QueryContext 查询多行

```go
type User struct {
	ID   int64
	Name string
	Age  int
}

func ListUsers(ctx context.Context, db *sql.DB, minAge int) ([]User, error) {
	rows, err := db.QueryContext(ctx,
		`SELECT id, name, age FROM users WHERE age >= ? ORDER BY id LIMIT 100`,
		minAge,
	)
	if err != nil {
		return nil, fmt.Errorf("query users: %w", err)
	}
	defer rows.Close()

	users := make([]User, 0)

	for rows.Next() {
		var u User
		if err := rows.Scan(&u.ID, &u.Name, &u.Age); err != nil {
			return nil, fmt.Errorf("scan user: %w", err)
		}
		users = append(users, u)
	}

	if err := rows.Err(); err != nil {
		return nil, fmt.Errorf("iterate users: %w", err)
	}

	return users, nil
}
```

---

## 【文本块 8】代码解释

这里有三个关键点。

第一，`defer rows.Close()` 必须写。
否则数据库连接可能无法归还连接池。

第二，循环结束后要检查 `rows.Err()`。
因为迭代过程中可能发生网络错误、连接错误、扫描错误。

第三，查询要带 LIMIT。
不要在业务接口里无脑全表查询，否则很容易导致 MySQL 和 Go 服务内存压力。

---

## 【代码块 4】QueryRowContext 查询单行

```go
func GetUserByID(ctx context.Context, db *sql.DB, id int64) (*User, error) {
	var u User

	err := db.QueryRowContext(ctx,
		`SELECT id, name, age FROM users WHERE id = ?`,
		id,
	).Scan(&u.ID, &u.Name, &u.Age)

	if err == sql.ErrNoRows {
		return nil, nil
	}
	if err != nil {
		return nil, fmt.Errorf("query user by id: %w", err)
	}

	return &u, nil
}
```

---

## 【文本块 9】追问：sql.ErrNoRows 是错误吗？

从技术上它是一个 error，但从业务语义上通常表示“数据不存在”。

不能直接当系统错误打 ERROR 日志。

一般处理方式：

```go
if err == sql.ErrNoRows {
    return nil, nil
}
```

或者包装成业务错误：

```go
return nil, ErrUserNotFound
```

在现代 Go 里，如果错误可能被包装，可以用：

```go
errors.Is(err, sql.ErrNoRows)
```

---

## 【代码块 5】ExecContext 更新数据

```go
func UpdateUserName(ctx context.Context, db *sql.DB, id int64, name string) error {
	result, err := db.ExecContext(ctx,
		`UPDATE users SET name = ?, updated_at = NOW() WHERE id = ?`,
		name,
		id,
	)
	if err != nil {
		return fmt.Errorf("update user name: %w", err)
	}

	affected, err := result.RowsAffected()
	if err != nil {
		return fmt.Errorf("get rows affected: %w", err)
	}

	if affected == 0 {
		return sql.ErrNoRows
	}

	return nil
}
```

---

# 四、Go 事务基础

## 【文本块 10】Q：Go 里怎么开启事务？

使用 `db.BeginTx` 开启事务：

```go
tx, err := db.BeginTx(ctx, &sql.TxOptions{})
```

事务里的所有 SQL 都必须使用 `tx.QueryContext`、`tx.ExecContext`、`tx.QueryRowContext`。

不能在事务里混用 `db.ExecContext`，否则那条 SQL 会走连接池里的其他连接，不属于当前事务。

事务结束时：

成功调用 `tx.Commit()`。
失败调用 `tx.Rollback()`。

常见模式是：

```go
tx, err := db.BeginTx(ctx, nil)
if err != nil {
    return err
}
defer tx.Rollback()

// do something

if err := tx.Commit(); err != nil {
    return err
}
```

为什么 defer Rollback 可以和 Commit 共存？

因为 Commit 成功后，再 Rollback 会返回错误，但通常可以忽略。它的作用是兜底，保证中途 return 时事务能回滚。

---

## 【代码块 6】Go 事务标准写法

```go
func Transfer(ctx context.Context, db *sql.DB, fromID, toID int64, amount int64) error {
	tx, err := db.BeginTx(ctx, &sql.TxOptions{
		Isolation: sql.LevelReadCommitted,
	})
	if err != nil {
		return fmt.Errorf("begin tx: %w", err)
	}
	defer tx.Rollback()

	if _, err := tx.ExecContext(ctx,
		`UPDATE accounts SET balance = balance - ? WHERE id = ? AND balance >= ?`,
		amount, fromID, amount,
	); err != nil {
		return fmt.Errorf("decrease balance: %w", err)
	}

	if _, err := tx.ExecContext(ctx,
		`UPDATE accounts SET balance = balance + ? WHERE id = ?`,
		amount, toID,
	); err != nil {
		return fmt.Errorf("increase balance: %w", err)
	}

	if err := tx.Commit(); err != nil {
		return fmt.Errorf("commit tx: %w", err)
	}

	return nil
}
```

---

## 【文本块 11】事务代码解释

这个例子里，扣款和加款必须放在同一个事务中。

如果扣款成功后加款失败，事务会回滚，避免钱凭空消失。

这里还有一个细节：

扣款 SQL 写成：

```sql
UPDATE accounts 
SET balance = balance - ? 
WHERE id = ? AND balance >= ?
```

这是为了在数据库层面保证余额不能扣成负数。

不要先 SELECT balance 到 Go 里判断，再 UPDATE。
因为高并发下，两个请求可能同时读到足够余额，然后都扣款，导致超扣。

更稳的做法是把条件放到 UPDATE 里，让数据库的行锁和原子更新保证并发安全。

---

## 【文本块 12】追问：事务里可以调用其他函数吗？

可以，但要注意事务对象必须往下传。

错误写法：

```go
func CreateOrder(ctx context.Context, db *sql.DB) error {
    tx, _ := db.BeginTx(ctx, nil)

    SaveOrder(ctx, db)
    SaveOrderItems(ctx, db)

    return tx.Commit()
}
```

这里 `SaveOrder` 用的是 `db`，不属于事务。

正确做法是抽象一个接口，或者显式传 `*sql.Tx`。

工程里常见做法是定义接口：

```go
type DBTX interface {
    ExecContext(ctx context.Context, query string, args ...any) (sql.Result, error)
    QueryContext(ctx context.Context, query string, args ...any) (*sql.Rows, error)
    QueryRowContext(ctx context.Context, query string, args ...any) *sql.Row
}
```

`*sql.DB` 和 `*sql.Tx` 都能满足这个接口。

---

## 【代码块 7】用 DBTX 统一 DB 和 Tx

```go
type DBTX interface {
	ExecContext(ctx context.Context, query string, args ...any) (sql.Result, error)
	QueryContext(ctx context.Context, query string, args ...any) (*sql.Rows, error)
	QueryRowContext(ctx context.Context, query string, args ...any) *sql.Row
}

type UserRepo struct {
	db DBTX
}

func NewUserRepo(db DBTX) *UserRepo {
	return &UserRepo{db: db}
}

func (r *UserRepo) CreateUser(ctx context.Context, name string, age int) error {
	_, err := r.db.ExecContext(ctx,
		`INSERT INTO users(name, age, created_at) VALUES (?, ?, NOW())`,
		name,
		age,
	)
	if err != nil {
		return fmt.Errorf("insert user: %w", err)
	}
	return nil
}
```

---

## 【文本块 13】Go 项目事务边界放在哪里？

一般放在 service 层，而不是 repo 层。

原因：

repo 层负责单表或单聚合的数据访问。
service 层负责业务流程，知道哪些操作必须在同一个事务里。

例如创建订单：

1. 创建订单
2. 创建订单明细
3. 扣库存
4. 写流水

这些操作是否在一个事务里，是业务流程决定的，不应该由某个单表 repo 私自决定。

面试回答：

Go 项目中事务边界通常放在 service 层。repo 层只提供基于 DBTX 的数据操作，既可以接收 `*sql.DB`，也可以接收 `*sql.Tx`。这样 service 可以在一个事务里组合多个 repo 操作，避免事务边界分散。

---

# 五、GORM 事务

## 【文本块 14】Q：GORM 里怎么使用事务？

GORM 提供两种事务写法。

第一种是手动事务：

```go
tx := db.Begin()
...
tx.Commit()
```

第二种是推荐的闭包事务：

```go
db.Transaction(func(tx *gorm.DB) error {
    return nil
})
```

闭包事务的好处是：

* 返回 nil 自动提交
* 返回 error 自动回滚
* panic 时自动回滚
* 代码更简洁

---

## 【代码块 8】GORM 闭包事务

```go
type Order struct {
	ID     int64
	UserID int64
	Amount int64
}

type Account struct {
	ID      int64
	Balance int64
}

func CreateOrderWithGORM(ctx context.Context, db *gorm.DB, order *Order) error {
	return db.WithContext(ctx).Transaction(func(tx *gorm.DB) error {
		if err := tx.Create(order).Error; err != nil {
			return fmt.Errorf("create order: %w", err)
		}

		err := tx.Model(&Account{}).
			Where("id = ? AND balance >= ?", order.UserID, order.Amount).
			UpdateColumn("balance", gorm.Expr("balance - ?", order.Amount)).
			Error
		if err != nil {
			return fmt.Errorf("decrease balance: %w", err)
		}

		return nil
	})
}
```

---

## 【文本块 15】GORM 事务注意事项

第一，一定要用 `tx`，不要在事务闭包里误用外层 `db`。

错误：

```go
db.Transaction(func(tx *gorm.DB) error {
    db.Create(&order)
    return nil
})
```

这里 `db.Create` 不在事务中。

正确：

```go
tx.Create(&order)
```

第二，必须传 context。

```go
db.WithContext(ctx)
```

否则请求取消或超时时，数据库操作可能还在继续。

第三，复杂 SQL 不要硬用 ORM 拼。
性能敏感路径、复杂 join、批量更新、锁查询，可以直接用 Raw SQL 或 `database/sql`。

---

# 六、事务 ACID

## 【文本块 16】Q：MySQL 事务的 ACID 是什么？

ACID 是事务的四个特性。

A：Atomicity，原子性。
一个事务中的操作要么全部成功，要么全部失败。比如转账，扣款和加款必须一起成功或一起回滚。InnoDB 主要通过 undo log 支持回滚，从而实现原子性。

C：Consistency，一致性。
事务执行前后，数据库要从一个一致状态转移到另一个一致状态。比如余额不能为负、订单状态不能乱跳、外键约束不能破坏。它依赖数据库约束、事务机制和业务逻辑共同保证。

I：Isolation，隔离性。
多个事务并发执行时，彼此之间不能随意干扰。隔离性由锁和 MVCC 共同实现。

D：Durability，持久性。
事务一旦提交，结果就要持久保存。InnoDB 主要通过 redo log 保证崩溃恢复。

面试里不要只背四个英文，要能对应到 InnoDB 机制：

原子性靠 undo log。
持久性靠 redo log。
隔离性靠 MVCC + 锁。
一致性是最终目标，依赖前三者和业务约束共同完成。

---

# 七、事务隔离级别

## 【文本块 17】Q：MySQL 四种隔离级别分别解决什么问题？

MySQL 事务隔离级别从低到高有四种：

第一，读未提交，Read Uncommitted。
可以读到其他事务还没提交的数据，会产生脏读。生产基本不用。

第二，读已提交，Read Committed，简称 RC。
只能读到其他事务已经提交的数据，解决脏读。但同一个事务内两次读取同一行，可能因为其他事务提交而结果不同，也就是不可重复读。

第三，可重复读，Repeatable Read，简称 RR。
MySQL InnoDB 默认隔离级别。同一个事务内，多次快照读看到的数据一致，解决不可重复读。InnoDB 在 RR 下还通过 Next-Key Lock 处理当前读场景下的幻读。

第四，串行化，Serializable。
所有事务近似串行执行，一致性最强，但性能最差，生产很少用。

并发问题对应：

脏读：读到别人未提交的数据。
不可重复读：同一事务内两次读同一行，值不同。
幻读：同一事务内两次范围查询，结果集行数不同。

---

## 【文本块 18】Q：RC 和 RR 有什么区别？

RC 和 RR 最大区别在于 Read View 的生成时机。

RC：每次快照读都会生成新的 Read View。
所以同一个事务里，第二次 SELECT 可以看到其他事务刚提交的数据。

RR：事务第一次快照读生成 Read View，后续复用同一个 Read View。
所以同一个事务里，多次 SELECT 看到的结果一致。

这就是为什么 RC 会有不可重复读，而 RR 可以解决不可重复读。

另外，在锁方面：

RR 下 InnoDB 更容易使用间隙锁和 Next-Key Lock，防止当前读出现幻读。
RC 下锁范围通常更小，并发能力更好，但幻读防护弱一些。

面试回答：

RC 每次读生成新的 Read View，能看到其他事务新提交的数据；RR 第一次读生成 Read View，后续复用，保证同一事务内快照一致。RR 一致性更强，但锁范围可能更大；RC 并发更好，很多互联网业务会选择 RC。

---

## 【文本块 19】为什么很多互联网公司喜欢 RC？

MySQL 默认是 RR，但很多线上业务会选择 RC。

原因是：

第一，RC 锁范围更小。
RR 下为了避免幻读，InnoDB 会使用更多间隙锁和 Next-Key Lock，可能扩大锁范围。

第二，RC 死锁和锁等待风险更低。
很多互联网业务更关心吞吐和可用性，而不是每个长事务内都必须看到完全一致的快照。

第三，业务通常可以接受读已提交语义。
很多查询是短事务或单语句读，不依赖 RR 的可重复读能力。

但这不是说 RC 一定比 RR 好。金融、库存、强一致状态机等场景，仍然要根据业务选择隔离级别。

一句话：

默认值不一定是线上最优值。RR 一致性更强，RC 并发性更好，具体选择要看业务对一致性和锁冲突的权衡。

---

# 八、MVCC

## 【文本块 20】Q：什么是 MVCC？

MVCC 全称 Multi-Version Concurrency Control，多版本并发控制。

它的核心思想是：

读不直接加锁阻塞写，而是通过数据的多个历史版本，让读事务看到符合自己可见性规则的版本。

这样可以提高并发性能。

InnoDB 中 MVCC 主要依赖三个东西：

第一，隐藏字段。
每行记录都有隐藏字段，比如：

* `DB_TRX_ID`：最近修改这行记录的事务 ID
* `DB_ROLL_PTR`：回滚指针，指向 undo log 中的旧版本

第二，undo log。
记录数据修改前的旧版本，形成版本链。

第三，Read View。
事务快照读时生成，用于判断版本链中哪个版本对当前事务可见。

面试回答：

MVCC 是 InnoDB 为了提升并发读写性能实现的多版本机制。它通过隐藏事务 ID、undo log 版本链和 Read View，让普通 SELECT 可以读取历史快照，而不必阻塞正在写入的事务。

---

## 【文本块 21】Q：Read View 是什么？

Read View 是事务执行快照读时生成的一份“活跃事务视图”。

它里面大致包含：

* 当前活跃事务 ID 列表
* 最小活跃事务 ID
* 下一个将要分配的事务 ID
* 当前事务 ID

判断某个版本是否可见时，会看该版本的 `trx_id`：

如果版本的事务 ID 小于最小活跃事务 ID，说明它在当前事务开始前已经提交，可见。

如果版本的事务 ID 大于等于下一个事务 ID，说明它是当前事务开始后才产生的，不可见。

如果版本的事务 ID 在活跃事务列表里，说明创建它的事务还没提交，不可见。

如果版本的事务 ID 不在活跃事务列表里，说明创建它的事务已经提交，可见。

面试不用把字段名字背得特别死，但要讲清楚本质：

Read View 的作用是判断版本链上的某个数据版本，对当前事务是否可见。

---

## 【文本块 22】Q：快照读和当前读有什么区别？

快照读：

普通 SELECT 通常是快照读。

例如：

```sql
SELECT * FROM users WHERE id = 1;
```

它通过 MVCC 读取符合当前 Read View 的历史版本，不加锁。

当前读：

当前读读取的是最新数据，并且通常会加锁。

例如：

```sql
SELECT * FROM users WHERE id = 1 FOR UPDATE;
SELECT * FROM users WHERE id = 1 LOCK IN SHARE MODE;
UPDATE users SET name = 'Tom' WHERE id = 1;
DELETE FROM users WHERE id = 1;
```

当前读必须读取最新版本，因为它要修改或锁定当前记录。

面试回答：

普通 SELECT 是快照读，读的是 MVCC 可见版本，不阻塞写；UPDATE、DELETE、SELECT FOR UPDATE 是当前读，读最新版本并加锁，用于保证后续修改的正确性。

---

# 九、幻读与 Next-Key Lock

## 【文本块 23】Q：InnoDB 如何解决幻读？

幻读是指同一个事务里，两次范围查询，结果集行数不一致。比如第一次查 age > 20 有 10 行，另一个事务插入一行 age = 25 并提交，第二次再查变成 11 行。

对于快照读，RR 下通过 MVCC 的 Read View 解决。
同一个事务复用同一个 Read View，看不到其他事务后插入的新数据。

对于当前读，必须靠锁解决。
如果只锁已有记录，其他事务仍然可以往范围间隙里插入新记录。于是 InnoDB 使用间隙锁和 Next-Key Lock。

Next-Key Lock = 记录锁 + 间隙锁。

它不仅锁住命中的记录，还锁住记录之间的间隙，防止其他事务在范围内插入新记录。

面试回答：

InnoDB 在 RR 下解决幻读分两种情况。普通 SELECT 通过 MVCC 快照读解决；当前读比如 SELECT FOR UPDATE，则通过 Next-Key Lock，也就是记录锁加间隙锁，锁住范围，防止其他事务插入新记录。

---

# 十、redo log、undo log、binlog

## 【文本块 24】Q：redo log、undo log、binlog 有什么区别？

这三个日志是 MySQL 面试的核心。

redo log 是 InnoDB 的物理日志。
它记录的是数据页做了什么修改，用于崩溃恢复，保证事务持久性。

undo log 是 InnoDB 的回滚日志。
它记录数据修改前的旧版本，用于事务回滚，也用于 MVCC 版本链。

binlog 是 MySQL Server 层的逻辑日志。
它记录逻辑变更，主要用于主从复制、数据恢复、审计和数据同步。

一句话记忆：

redo log：向前恢复。
undo log：向后回滚。
binlog：复制和归档。

面试回答：

redo log 是 InnoDB 用来崩溃恢复的物理日志；undo log 记录旧版本，用于回滚和 MVCC；binlog 是 Server 层逻辑日志，用于主从复制和数据恢复。三者职责不同，不能混为一谈。

---

## 【文本块 25】Q：为什么有了 binlog 还需要 redo log？

因为它们解决的问题不同。

binlog 是 Server 层日志，偏逻辑变更，比如某行被更新成什么值，主要用于复制和恢复。

redo log 是 InnoDB 层日志，记录数据页物理修改，用于崩溃恢复。

如果没有 redo log，事务提交后数据页还没刷盘时 MySQL 崩溃，就可能丢失已经提交的数据。

redo log 采用 WAL 思想，也就是 Write-Ahead Logging：

先写日志，再刷数据页。

这样事务提交时不必立刻把所有数据页刷盘，只要 redo log 持久化了，即使崩溃也能恢复。

---

## 【文本块 26】Q：两阶段提交是什么？为什么需要？

MySQL 更新事务提交时，涉及两个日志：

* InnoDB 的 redo log
* Server 层的 binlog

如果这两个日志不一致，就会出问题。

比如 redo log 提交了，但 binlog 没写成功。
那么主库崩溃恢复后有这条数据，但从库通过 binlog 复制时没有这条数据，主从不一致。

所以 MySQL 使用两阶段提交：

第一阶段，prepare。
先写 redo log，并标记为 prepare 状态。

第二阶段，commit。
写 binlog。binlog 写成功后，再把 redo log 标记为 commit。

崩溃恢复时：

如果 redo log 是 prepare，但没有对应 binlog，说明事务没有真正提交，回滚。
如果 redo log 是 prepare，并且有对应 binlog，说明事务应该提交，可以恢复提交。
如果 redo log 是 commit，事务已经提交。

面试回答：

两阶段提交是为了保证 redo log 和 binlog 的一致性。先把 redo log 写成 prepare，再写 binlog，最后提交 redo log。这样崩溃恢复时可以根据 redo log 状态和 binlog 是否存在判断事务应该提交还是回滚，避免主从复制和崩溃恢复不一致。

---

# 十一、一条 SQL 的执行流程

## 【文本块 27】Q：一条 SELECT 语句在 MySQL 中怎么执行？

一条查询 SQL 大致流程是：

第一，客户端和 MySQL 建立连接。
连接器负责认证、权限校验、连接管理。

第二，解析 SQL。
解析器做词法和语法分析，把 SQL 转成语法树。

第三，预处理和权限检查。
检查表、字段是否存在，权限是否足够。

第四，优化器生成执行计划。
决定使用哪个索引、表连接顺序、是否排序、是否临时表等。

第五，执行器调用存储引擎。
执行器根据执行计划调用 InnoDB 接口读取数据。

第六，返回结果。
MySQL 把结果集返回给客户端。

面试回答：

查询语句的执行过程可以概括为：连接器建立连接和鉴权，解析器解析 SQL，优化器选择执行计划，执行器调用存储引擎读取数据，最后返回结果。

---

## 【文本块 28】Q：一条 UPDATE 语句怎么执行？

UPDATE 比 SELECT 多了日志和事务流程。

例如：

```sql
UPDATE users SET name = 'Tom' WHERE id = 1;
```

大致流程：

1. 连接器、解析器、优化器、执行器前面流程类似
2. InnoDB 根据索引找到要更新的记录
3. 加锁
4. 生成 undo log，保存旧版本
5. 修改内存中的数据页
6. 写 redo log prepare
7. Server 层写 binlog
8. redo log commit
9. 事务提交后，后台再把脏页刷盘

这里体现了 WAL 思想：

事务提交不需要立刻把数据页刷到磁盘，只要 redo log 落盘，就能保证崩溃恢复。

---

# 十二、大事务

## 【文本块 29】Q：大事务有什么问题？如何优化？

大事务是生产环境常见问题。

它的问题主要有：

第一，锁持有时间长。
大事务修改大量数据，持有很多行锁甚至间隙锁，其他事务会被阻塞。

第二，undo log 膨胀。
事务未提交前，旧版本不能释放，undo log 会越来越大。

第三，主从延迟。
大事务生成大量 binlog，从库重放需要很久。

第四，回滚成本高。
大事务如果失败，回滚要撤销大量修改，可能比执行还慢。

第五，影响 MVCC。
长事务会导致历史版本无法清理，影响 purge，造成 undo 堆积。

优化方式：

* 分批处理
* 缩短事务时间
* 不要在事务里做 RPC、HTTP、Redis 等外部慢操作
* 避免一次性更新大量行
* 避免长时间打开事务不提交
* 大批量任务放到后台，控制批次和间隔
* 必要时使用任务表记录进度，支持断点续跑

面试回答：

大事务的核心问题是持锁久、undo log 大、回滚慢、主从延迟高。优化思路是拆成小事务，缩短事务范围，避免事务里做外部 IO，批量任务分批提交。

---

# 十三、索引原理

## 【文本块 30】Q：MySQL 为什么用 B+ 树做索引？

MySQL InnoDB 常用 B+ 树作为索引结构。

B+ 树相比二叉树、红黑树更适合磁盘数据库，核心原因是减少磁盘 IO。

第一，B+ 树是多叉树。
每个节点可以存很多 key，所以树高比较低。树高低意味着查询时访问磁盘页次数少。

第二，非叶子节点只存索引 key，不存完整数据。
这样一个页能放更多 key，进一步降低树高。

第三，叶子节点之间用链表连接。
范围查询很方便，比如 `BETWEEN`、`ORDER BY`、分页扫描。

第四，所有数据都在叶子节点。
查询路径稳定。

面试回答：

B+ 树适合 MySQL 索引，是因为它多叉、树高低、磁盘 IO 少，叶子节点有序链表支持范围查询，非叶子节点只存 key 能提高扇出。它的核心优势是用更少的磁盘 IO 定位数据。

---

## 【文本块 31】Q：聚簇索引和二级索引有什么区别？

InnoDB 表的数据本身就是按主键组织的一棵 B+ 树，这棵树叫聚簇索引。

聚簇索引的叶子节点存的是整行数据。

二级索引，也叫普通索引或辅助索引。
它的叶子节点存的是索引列 + 主键值。

所以通过二级索引查数据时，如果要的字段不在二级索引里，就需要先从二级索引找到主键，再回到聚簇索引查整行数据。这个过程叫回表。

例如：

```sql
CREATE INDEX idx_name ON users(name);

SELECT age FROM users WHERE name = 'Tom';
```

如果 idx_name 只包含 name 和主键 id，而 age 不在索引里，就要回表查 age。

面试回答：

聚簇索引叶子节点存整行数据，InnoDB 的主键索引就是聚簇索引；二级索引叶子节点存索引列和主键，如果查询字段不在二级索引里，就要通过主键回到聚簇索引查整行，这就是回表。

---

## 【文本块 32】Q：什么是覆盖索引？

覆盖索引是指查询需要的字段都能从索引中直接拿到，不需要回表。

例如有联合索引：

```sql
CREATE INDEX idx_name_age ON users(name, age);
```

查询：

```sql
SELECT age FROM users WHERE name = 'Tom';
```

这个查询只需要 name 和 age，都在索引 idx_name_age 里，就不需要回表。

覆盖索引的好处：

* 减少回表
* 减少随机 IO
* 提高查询性能
* 对高频查询特别有价值

但不能为了覆盖所有查询，把很多字段都塞进联合索引。索引越多、越大，写入成本越高，占用空间越多。

面试回答：

覆盖索引就是查询字段都在索引里，能直接从索引返回结果，避免回表。它可以显著减少 IO，但设计时要权衡查询收益、索引大小和写入维护成本。

---

# 十四、联合索引与最左前缀

## 【文本块 33】Q：联合索引的最左前缀原则是什么？

假设有联合索引：

```sql
CREATE INDEX idx_abc ON t(a, b, c);
```

这个索引是先按 a 排序，a 相同再按 b 排序，b 相同再按 c 排序。

所以它能高效支持：

```sql
WHERE a = 1
WHERE a = 1 AND b = 2
WHERE a = 1 AND b = 2 AND c = 3
```

但不能高效支持：

```sql
WHERE b = 2
WHERE c = 3
```

因为缺少最左列 a，无法利用索引的整体有序性。

范围查询也会影响后续列的使用：

```sql
WHERE a = 1 AND b > 2 AND c = 3
```

通常 a 和 b 可以用于索引定位，但 c 很难继续用于有序定位，因为 b 是范围条件，范围内 c 不再整体有序。

面试回答：

联合索引按照从左到右的列顺序排序，查询必须从最左列开始连续匹配，才能利用索引有序性。范围查询后面的列通常不能继续用于索引定位，但可能通过索引下推减少回表。

---

## 【文本块 34】Q：设计联合索引时，第一列一定放区分度最高的吗？

不一定。

这是一个常见误区。

联合索引列顺序要综合考虑：

第一，查询条件出现频率。
最常用的过滤条件应该优先考虑。

第二，等值条件优先。
等值条件适合放在范围条件前面。

第三，排序和分组需求。
如果 ORDER BY、GROUP BY 能利用索引顺序，可以减少 filesort。

第四，区分度。
区分度高的列通常过滤效果好，但不是唯一标准。

第五，覆盖索引可能性。
高频查询是否能通过联合索引覆盖。

例如：

```sql
WHERE tenant_id = ? AND status = ? AND created_at BETWEEN ? AND ?
ORDER BY created_at DESC
```

在多租户系统里，tenant_id 可能区分度不是最高，但几乎所有查询都带它，而且是数据隔离条件，所以通常应该放前面。

面试回答：

联合索引不是简单把区分度最高的列放最左，而是要结合查询频率、等值匹配、范围条件、排序需求和覆盖索引综合设计。

---

# 十五、索引失效

## 【文本块 35】Q：哪些情况会导致索引失效？

常见索引失效场景：

第一，对索引列做函数运算。

```sql
WHERE YEAR(create_time) = 2026
```

这样 MySQL 很难利用 create_time 的有序性。

应该改成：

```sql
WHERE create_time >= '2026-01-01' 
  AND create_time < '2027-01-01'
```

第二，隐式类型转换。

字段是 varchar，但查询写：

```sql
WHERE phone = 13800138000
```

可能导致类型转换，影响索引。

第三，LIKE 前导通配符。

```sql
WHERE name LIKE '%tom'
```

因为不知道从哪里开始匹配，无法利用 B+ 树前缀有序性。

第四，OR 条件不当。

如果 OR 两边不是都能用索引，可能导致全表扫描。

第五，范围查询后面的联合索引列无法继续用于定位。

第六，低选择性条件可能被优化器放弃索引。

例如性别、布尔字段，如果匹配大量行，走索引再回表可能不如全表扫描。

第七，使用 !=、<>、IS NOT NULL。
不是绝对不能用索引，但如果匹配行太多，优化器可能选择全表扫描。

面试回答：

索引失效的本质是查询条件无法利用索引的有序性，或者优化器认为走索引成本更高。常见原因包括函数运算、隐式转换、LIKE 前导通配符、OR 条件、范围查询截断、低选择性条件等。

---

# 十六、EXPLAIN 与 SQL 优化

## 【文本块 36】Q：怎么分析 MySQL 执行计划？

最常用的是 EXPLAIN。

重点关注字段：

`type`：访问类型。
常见从好到差大致有 const、eq_ref、ref、range、index、ALL。ALL 通常表示全表扫描，要警惕。

`key`：实际使用的索引。
如果 key 为 NULL，说明没用上索引。

`rows`：预估扫描行数。
rows 越大，说明扫描成本越高。

`Extra`：额外信息。
重点看：

* Using index：覆盖索引
* Using where：Server 层过滤
* Using index condition：索引下推
* Using filesort：额外排序
* Using temporary：使用临时表

面试回答：

我看 EXPLAIN 主要关注 type、key、rows 和 Extra。type 看访问路径是否高效，key 看实际索引，rows 看扫描量，Extra 看是否有 filesort、temporary、回表和索引下推。SQL 优化要结合执行计划，而不是只看 SQL 表面。

---

## 【文本块 37】SQL 优化一般怎么做？

我一般按这个顺序排查：

第一，看慢查询日志。
先确认到底哪条 SQL 慢，不凭感觉优化。

第二，看 EXPLAIN。
确认是否走索引、扫描行数、是否 filesort、temporary。

第三，看索引设计。
联合索引顺序是否合理，是否满足最左前缀，是否能覆盖。

第四，看 SQL 写法。
是否函数操作索引列、隐式类型转换、select *、深分页、OR 条件。

第五，看返回数据量。
很多时候 SQL 慢不是因为索引，而是一次返回太多行。

第六，看表结构和数据分布。
字段选择性、数据倾斜、统计信息是否准确。

第七，看锁等待。
SQL 慢可能不是查询本身慢，而是在等锁。

第八，看系统资源。
CPU、IO、buffer pool、连接数、主从延迟。

面试回答：

SQL 优化不是上来就加索引，而是先定位慢 SQL，再看执行计划，分析索引、回表、排序、临时表、扫描行数和锁等待，最后结合业务访问模式调整 SQL 和索引。

---

# 十七、深分页

## 【文本块 38】Q：MySQL 深分页为什么慢？怎么优化？

典型深分页：

```sql
SELECT * FROM orders ORDER BY id LIMIT 1000000, 20;
```

MySQL 需要先扫描并丢弃前 1000000 行，再返回 20 行。

即使走索引，也要扫描大量索引记录，如果还要回表，成本更高。

优化方式：

第一，基于游标分页。

```sql
SELECT * FROM orders 
WHERE id > ? 
ORDER BY id 
LIMIT 20;
```

适合按递增主键翻页。

第二，延迟关联。

先从索引里查出主键，再回表查完整数据：

```sql
SELECT o.*
FROM orders o
JOIN (
    SELECT id FROM orders ORDER BY id LIMIT 1000000, 20
) t ON o.id = t.id;
```

第三，限制最大翻页深度。
很多业务没必要允许用户翻到第 10000 页。

第四，使用搜索引擎或离线方案。
复杂搜索和深分页可以考虑 Elasticsearch。

面试回答：

深分页慢是因为 offset 越大，需要扫描并丢弃的数据越多。优化思路是改成基于游标的 seek pagination，或者先用覆盖索引查主键再回表，必要时限制最大页数。

---

## 【代码块 9】Go 游标分页

```go
func ListOrdersAfterID(ctx context.Context, db *sql.DB, lastID int64, limit int) ([]Order, error) {
	rows, err := db.QueryContext(ctx,
		`SELECT id, user_id, amount, created_at
		 FROM orders
		 WHERE id > ?
		 ORDER BY id
		 LIMIT ?`,
		lastID,
		limit,
	)
	if err != nil {
		return nil, fmt.Errorf("query orders after id: %w", err)
	}
	defer rows.Close()

	orders := make([]Order, 0, limit)

	for rows.Next() {
		var o Order
		if err := rows.Scan(&o.ID, &o.UserID, &o.Amount, &o.CreatedAt); err != nil {
			return nil, fmt.Errorf("scan order: %w", err)
		}
		orders = append(orders, o)
	}

	if err := rows.Err(); err != nil {
		return nil, fmt.Errorf("iterate orders: %w", err)
	}

	return orders, nil
}
```

---

# 十八、锁机制

## 【文本块 39】Q：MySQL 为什么需要锁？

数据库锁的本质是控制并发访问顺序，保证数据一致性。

如果多个事务同时读写同一份数据，没有锁或 MVCC，就会出现：

* 丢失更新
* 脏读
* 不可重复读
* 幻读
* 数据状态错乱

InnoDB 的并发控制主要是两条线：

普通读尽量通过 MVCC，不加锁，提高并发。
写和当前读通过锁控制顺序，保证一致性。

面试回答：

MySQL 锁的本质是协调并发事务对共享数据的访问顺序。MVCC 解决读写并发，锁主要解决写写冲突、当前读和范围插入问题。

---

## 【文本块 40】Q：MySQL 常见锁有哪些？

从粒度看：

表锁：锁整张表。粒度大，并发差，但实现简单。
行锁：锁具体记录。粒度小，并发好，是 InnoDB 常用锁。
间隙锁：锁记录之间的间隙，防止范围内插入新记录。
Next-Key Lock：记录锁 + 间隙锁。

从语义看：

共享锁，S 锁。
多个事务可以同时加共享锁读，但会阻塞写。

排他锁，X 锁。
写锁，其他事务通常不能再读写同一资源。

面试回答：

MySQL 锁可以从粒度和语义两条线讲。粒度上有表锁、行锁、间隙锁、Next-Key Lock；语义上有共享锁和排他锁。InnoDB 的重点是行锁和 Next-Key Lock。

---

## 【文本块 41】Q：InnoDB 行锁是锁物理行吗？

不是简单锁物理行，而是锁索引记录。

这点非常重要。

如果 SQL 命中索引，InnoDB 可以精准锁定索引记录。

如果 SQL 没走索引，InnoDB 可能扫描更多记录，导致锁范围扩大，甚至看起来像锁了整张表。

例如：

```sql
UPDATE users SET status = 1 WHERE phone = '138...';
```

如果 phone 没有索引，MySQL 可能全表扫描，扫描过程中对大量记录加锁，锁冲突会非常严重。

面试回答：

InnoDB 行锁是基于索引实现的。SQL 走索引时锁范围更精准；不走索引时可能扫描大量记录并加锁，导致锁范围扩大。所以很多锁冲突问题，本质上是索引设计问题。

---

## 【文本块 42】Q：MySQL 为什么会死锁？怎么避免？

死锁是多个事务互相等待对方持有的锁，形成循环依赖。

典型例子：

事务 A 先锁订单 1，再锁订单 2。
事务 B 先锁订单 2，再锁订单 1。
两边互相等待，形成死锁。

InnoDB 可以检测死锁，并主动回滚其中一个事务。

避免死锁的常见方式：

第一，固定访问顺序。
多个事务更新多行数据时，按相同顺序更新，比如都按 id 从小到大。

第二，缩短事务时间。
不要在事务里做 RPC、HTTP、Redis、复杂计算。

第三，保证 SQL 走索引。
减少锁范围。

第四，减少批量大事务。
分批提交。

第五，必要时捕获死锁错误并重试。
死锁在高并发下无法百分百避免，业务要具备重试能力。

---

## 【代码块 10】Go 中对死锁做有限重试

```go
func WithRetry(ctx context.Context, maxRetry int, fn func() error) error {
	var lastErr error

	for i := 0; i <= maxRetry; i++ {
		if err := fn(); err != nil {
			lastErr = err

			if IsMySQLDeadlock(err) {
				time.Sleep(time.Duration(i+1) * 50 * time.Millisecond)
				continue
			}

			return err
		}

		return nil
	}

	return fmt.Errorf("retry exhausted: %w", lastErr)
}

func IsMySQLDeadlock(err error) bool {
	// go-sql-driver/mysql 中可以断言 *mysql.MySQLError
	// 1213: Deadlock found when trying to get lock
	// 1205: Lock wait timeout exceeded
	return false
}
```

---

## 【文本块 43】工程提醒

死锁重试不是万能药。

如果业务操作不是幂等的，盲目重试可能造成重复扣款、重复发货、重复写消息。

所以重试必须配合：

* 幂等 key
* 唯一约束
* 状态机
* 事务边界
* 去重表
* 请求 ID

面试里可以补一句：

死锁可以做有限重试，但前提是业务操作可重试、可幂等，否则重试会放大问题。

---

# 十九、SELECT FOR UPDATE

## 【文本块 44】Q：SELECT FOR UPDATE 有什么用？

`SELECT FOR UPDATE` 是当前读，会读取最新数据并加排他锁。

常用于：

* 查询后马上更新
* 库存扣减
* 余额变更
* 状态机流转
* 防止并发修改

例如：

```sql
SELECT stock FROM products WHERE id = ? FOR UPDATE;
```

这会锁住对应记录，其他事务想修改这行需要等待。

但要注意：

必须在事务中使用才有意义。
必须走索引，否则可能锁范围很大。
事务不能太长，否则锁持有时间过久。
在 RC 和 RR 下锁行为可能有差异。

---

## 【代码块 11】Go 中使用 SELECT FOR UPDATE

```go
func DeductStock(ctx context.Context, db *sql.DB, productID int64, count int64) error {
	tx, err := db.BeginTx(ctx, &sql.TxOptions{
		Isolation: sql.LevelReadCommitted,
	})
	if err != nil {
		return fmt.Errorf("begin tx: %w", err)
	}
	defer tx.Rollback()

	var stock int64
	err = tx.QueryRowContext(ctx,
		`SELECT stock FROM products WHERE id = ? FOR UPDATE`,
		productID,
	).Scan(&stock)
	if err != nil {
		return fmt.Errorf("select stock for update: %w", err)
	}

	if stock < count {
		return fmt.Errorf("stock not enough")
	}

	_, err = tx.ExecContext(ctx,
		`UPDATE products SET stock = stock - ? WHERE id = ?`,
		count,
		productID,
	)
	if err != nil {
		return fmt.Errorf("update stock: %w", err)
	}

	if err := tx.Commit(); err != nil {
		return fmt.Errorf("commit tx: %w", err)
	}

	return nil
}
```

---

## 【文本块 45】更推荐的库存扣减写法

很多场景下，不一定要先 SELECT FOR UPDATE 再 UPDATE。

可以直接写成条件 UPDATE：

```sql
UPDATE products
SET stock = stock - ?
WHERE id = ? AND stock >= ?
```

然后检查 affected rows。

这样更短、更直接，也能利用 InnoDB 的行锁保证原子性。

---

## 【代码块 12】条件 UPDATE 扣库存

```go
func DeductStockAtomic(ctx context.Context, db *sql.DB, productID int64, count int64) error {
	result, err := db.ExecContext(ctx,
		`UPDATE products
		 SET stock = stock - ?
		 WHERE id = ? AND stock >= ?`,
		count,
		productID,
		count,
	)
	if err != nil {
		return fmt.Errorf("deduct stock: %w", err)
	}

	affected, err := result.RowsAffected()
	if err != nil {
		return fmt.Errorf("rows affected: %w", err)
	}

	if affected == 0 {
		return fmt.Errorf("stock not enough")
	}

	return nil
}
```

---

# 二十、乐观锁

## 【文本块 46】Q：MySQL 乐观锁怎么实现？

乐观锁通常通过 version 字段实现。

表里加一个字段：

```sql
version int not null
```

更新时带上旧 version：

```sql
UPDATE orders
SET status = ?, version = version + 1
WHERE id = ? AND version = ?
```

如果 affected rows 是 1，说明更新成功。
如果是 0，说明数据已经被别人改过，需要重试或返回冲突。

乐观锁适合：

* 冲突不高
* 不想长时间持锁
* 状态更新
* 配置更新
* 防止覆盖别人修改

不适合：

* 冲突特别高
* 更新逻辑很复杂
* 必须严格串行的场景

---

## 【代码块 13】Go 实现乐观锁更新

```go
func UpdateOrderStatusOptimistic(
	ctx context.Context,
	db *sql.DB,
	orderID int64,
	oldVersion int64,
	newStatus string,
) error {
	result, err := db.ExecContext(ctx,
		`UPDATE orders
		 SET status = ?, version = version + 1, updated_at = NOW()
		 WHERE id = ? AND version = ?`,
		newStatus,
		orderID,
		oldVersion,
	)
	if err != nil {
		return fmt.Errorf("update order optimistic: %w", err)
	}

	affected, err := result.RowsAffected()
	if err != nil {
		return fmt.Errorf("rows affected: %w", err)
	}

	if affected == 0 {
		return fmt.Errorf("order version conflict")
	}

	return nil
}
```

---

# 二十一、主从复制

## 【文本块 47】Q：MySQL 主从复制原理是什么？

MySQL 主从复制基于 binlog。

经典流程：

第一，主库提交事务时写 binlog。

第二，从库 IO 线程连接主库，拉取 binlog。

第三，从库把拉到的 binlog 写入 relay log。

第四，从库 SQL 线程读取 relay log，并重放其中的变更。

所以主从复制本质是：

主库记录逻辑变更，从库重放这些变更。

默认情况下，MySQL 主从复制通常是异步复制。主库提交成功时，不一定等从库也执行完成。

这就会产生主从延迟。

---

## 【文本块 48】Q：为什么会出现主从延迟？

常见原因：

第一，主库写入压力大。
binlog 产生速度超过从库消费速度。

第二，从库 SQL 线程重放慢。
尤其是大事务、复杂 DDL、大批量更新。

第三，从库机器配置差。
CPU、IO、内存不如主库。

第四，从库同时承担大量读请求。
读请求占用资源，影响复制回放。

第五，网络延迟。
跨机房复制更明显。

第六，锁等待。
从库回放 SQL 时也可能遇到锁等待。

面试回答：

主从延迟本质是从库重放 relay log 的速度跟不上主库产生 binlog 的速度。大事务、从库负载高、机器性能差、网络延迟、锁等待都可能导致延迟。

---

## 【文本块 49】主从延迟会带来什么问题？

最常见问题是“写后读不一致”。

比如：

1. 用户修改昵称，请求写主库成功
2. 下一次查询走从库
3. 从库还没同步到新昵称
4. 用户看到旧昵称

解决方案：

第一，写后短时间强制读主。
比如更新用户资料后，几秒内用户自己的查询走主库。

第二，根据业务路由。
强一致读走主库，普通列表页走从库。

第三，半同步复制。
主库至少等待一个从库确认收到日志再返回，但会影响写入性能。

第四，查询从库延迟指标。
延迟过高时临时摘除从库或读主。

第五，业务层做最终一致提示。
比如“资料已更新，稍后生效”。

面试回答：

主从延迟最直接的问题是写后读旧数据。Go 服务里可以通过强一致读走主库、写后短时间读主、监控从库延迟和降级摘除从库来处理。

---

# 二十二、Go 读写分离

## 【文本块 50】Go 项目里怎么做读写分离？

最简单方式是维护两个 DB：

```go
masterDB *sql.DB
slaveDB  *sql.DB
```

写操作走 master。
普通读走 slave。
强一致读走 master。

更复杂场景下，可以有多个从库：

```go
slaves []*sql.DB
```

读请求通过轮询、随机、权重、延迟指标选择一个从库。

但读写分离不能无脑做。

这些场景应该读主：

* 写后立刻读
* 订单支付状态
* 库存扣减结果
* 用户刚修改的资料
* 强一致业务判断
* 事务内读取

---

## 【代码块 14】简单读写分离封装

```go
type DBRouter struct {
	Master *sql.DB
	Slaves []*sql.DB
	next   uint64
}

func (r *DBRouter) WriteDB() *sql.DB {
	return r.Master
}

func (r *DBRouter) ReadDB(strong bool) *sql.DB {
	if strong || len(r.Slaves) == 0 {
		return r.Master
	}

	idx := atomic.AddUint64(&r.next, 1)
	return r.Slaves[int(idx)%len(r.Slaves)]
}
```

---

## 【文本块 51】读写分离工程注意点

第一，事务必须走同一个连接。
事务里不能读从库、写主库混用。

第二，写后读要读主。
否则用户可能看到旧数据。

第三，要监控从库延迟。
延迟过高的从库不能继续承担读流量。

第四，连接池要分开配置。
主库和从库的连接池应该分别限制，避免读流量影响写库。

第五，要有降级策略。
从库故障时，是读主库还是返回错误，要看业务和主库压力。

---

# 二十三、InnoDB 和 MyISAM

## 【文本块 52】Q：InnoDB 和 MyISAM 有什么区别？

面试里抓核心差异即可。

第一，InnoDB 支持事务，MyISAM 不支持事务。

第二，InnoDB 支持行锁，MyISAM 主要是表锁。

第三，InnoDB 支持 MVCC，MyISAM 不支持。

第四，InnoDB 有 redo log、undo log，崩溃恢复能力强。

第五，InnoDB 支持外键，MyISAM 不支持。

第六，InnoDB 是现在 MySQL 默认和主流存储引擎。

线上核心业务表基本都应该用 InnoDB。

面试回答：

InnoDB 支持事务、行锁、MVCC、崩溃恢复和外键；MyISAM 不支持事务，主要是表锁，崩溃恢复能力弱。现在核心业务表基本都会选 InnoDB。

---

# 二十四、GORM、sqlx、手写 SQL 怎么选？

## 【文本块 53】Q：Go 项目里用 GORM 还是手写 SQL？

这要看场景。

GORM 优点：

* 开发快
* CRUD 简单
* 自动映射结构体
* 支持 hook、association、migration
* 对后台管理类系统很方便

GORM 缺点：

* 复杂 SQL 可读性差
* 容易生成不符合预期的 SQL
* 性能敏感路径不够直观
* 容易 N+1 查询
* 调试时需要看实际 SQL

sqlx 优点：

* 保留 SQL 可控性
* 支持结构体扫描
* 比 database/sql 更方便
* 适合中大型业务服务

手写 database/sql 优点：

* 最可控
* 依赖少
* 性能透明
* 适合基础设施和核心高性能路径

我的工程取舍：

普通 CRUD、后台管理、迭代速度优先，可以用 GORM。
复杂查询、核心链路、性能敏感、需要精确控制 SQL，用 sqlx 或 database/sql。
无论用 ORM 还是手写 SQL，都必须关注实际 SQL、索引和执行计划。

---

## 【文本块 54】GORM 常见坑

第一，N+1 查询。
循环里查关联数据，导致大量 SQL。

第二，Save 全字段更新。
可能把零值也更新到数据库，导致误覆盖。

第三，Preload 使用不当。
关联数据太多，一次查爆内存。

第四，事务里误用外层 db。
导致部分 SQL 不在事务中。

第五，忽略 RowsAffected。
更新失败但代码以为成功。

第六，不看实际 SQL。
ORM 生成的 SQL 可能没走索引。

第七，自动迁移滥用。
生产环境不要无脑 AutoMigrate 改表结构。

---

# 二十五、SQL 注入

## 【文本块 55】Q：Go 里如何防止 SQL 注入？

核心原则：不要拼接用户输入到 SQL 字符串里。

错误写法：

```go
sql := fmt.Sprintf("SELECT * FROM users WHERE name = '%s'", name)
```

如果 name 是恶意输入，就可能注入。

正确写法是使用参数占位符：

```go
db.QueryContext(ctx, "SELECT * FROM users WHERE name = ?", name)
```

驱动会把参数和 SQL 结构分开处理，避免注入。

GORM 也要用参数绑定：

```go
db.Where("name = ?", name).Find(&users)
```

不能写：

```go
db.Where("name = " + name).Find(&users)
```

面试回答：

防 SQL 注入的核心是参数化查询，不把用户输入直接拼进 SQL。无论 database/sql、sqlx 还是 GORM，都要使用占位符和参数绑定。

---

# 二十六、唯一约束与幂等

## 【文本块 56】Q：高并发下如何防止重复创建订单？

最稳的方式之一是数据库唯一约束。

比如订单表里加唯一索引：

```sql
UNIQUE KEY uk_request_id (request_id)
```

每个请求带一个唯一 request_id。

插入订单时，如果重复请求进来，第二次插入会违反唯一约束。

这比只依赖 Redis 锁更可靠。

Redis 锁可能过期、误删、主从切换不一致；数据库唯一约束是最终落库层面的强约束。

Go 代码里要识别 duplicate key 错误，把它转换成幂等成功或业务冲突。

---

## 【代码块 15】用唯一约束保证幂等

```go
func CreateOrderIdempotent(ctx context.Context, db *sql.DB, requestID string, userID int64, amount int64) error {
	_, err := db.ExecContext(ctx,
		`INSERT INTO orders(request_id, user_id, amount, created_at)
		 VALUES (?, ?, ?, NOW())`,
		requestID,
		userID,
		amount,
	)
	if err != nil {
		if IsDuplicateKey(err) {
			return nil
		}
		return fmt.Errorf("insert order: %w", err)
	}

	return nil
}

func IsDuplicateKey(err error) bool {
	// go-sql-driver/mysql duplicate entry error code: 1062
	return false
}
```

---

## 【文本块 57】面试补充

幂等不要只放在应用内存里，也不要只靠 Redis。

最稳的是：

* 请求唯一 ID
* 数据库唯一约束
* 状态机判断
* 事务内写入
* 必要时配合 Redis 防重复提交

数据库约束是最后防线。

---

# 二十七、MySQL Go 排障

## 【文本块 58】Q：Go 服务里 MySQL 慢，怎么排查？

我会按下面顺序排查：

第一，看 Go 服务侧指标。

* 请求 RT
* DB 调用耗时
* DB 错误率
* 连接池等待数
* open connections
* in use connections
* idle connections

`database/sql` 提供了：

```go
db.Stats()
```

可以查看连接池状态。

第二，看 MySQL 慢查询日志。
确认具体慢 SQL。

第三，看 EXPLAIN。
分析索引、扫描行数、filesort、temporary。

第四，看锁等待。
如果 SQL 本身不慢，但执行时间很长，可能是在等锁。

第五，看主从延迟。
如果读请求慢，可能从库延迟或压力过大。

第六，看资源指标。
CPU、IO、buffer pool 命中率、连接数、磁盘延迟。

第七，看最近变更。
是否刚上线新 SQL、新索引、表结构变更、批量任务。

面试回答：

Go 服务 MySQL 慢，我会先看服务侧 DB 调用耗时和连接池状态，再看 MySQL 慢查询和执行计划，然后排查索引、锁等待、主从延迟和数据库资源。不要一上来就加连接池或加索引。

---

## 【代码块 16】打印 database/sql 连接池指标

```go
func PrintDBStats(db *sql.DB) {
	stats := db.Stats()

	fmt.Println("MaxOpenConnections:", stats.MaxOpenConnections)
	fmt.Println("OpenConnections:", stats.OpenConnections)
	fmt.Println("InUse:", stats.InUse)
	fmt.Println("Idle:", stats.Idle)
	fmt.Println("WaitCount:", stats.WaitCount)
	fmt.Println("WaitDuration:", stats.WaitDuration)
	fmt.Println("MaxIdleClosed:", stats.MaxIdleClosed)
	fmt.Println("MaxIdleTimeClosed:", stats.MaxIdleTimeClosed)
	fmt.Println("MaxLifetimeClosed:", stats.MaxLifetimeClosed)
}
```

---

## 【文本块 59】连接池 WaitCount 很高说明什么？

`WaitCount` 很高说明很多请求在等待数据库连接。

可能原因：

* `MaxOpenConns` 太小
* SQL 太慢，连接长期被占用
* 事务太长
* MySQL 慢导致连接归还慢
* 请求量超过数据库承载能力

不能简单看到 WaitCount 高就把连接池调大。

如果 SQL 慢，调大连接池只会让更多请求同时压到 MySQL，可能导致雪崩。

正确做法是同时看：

* SQL 耗时
* 慢查询
* MySQL CPU/IO
* 连接池 InUse
* 请求 QPS
* 事务时长

---

# 二十八、本章高频速记

## 【文本块 60】Go MySQL 接入速记

`database/sql` 是标准抽象。
`go-sql-driver/mysql` 是常用 MySQL 驱动。
`sql.DB` 是连接池，不是单连接。
`sql.DB` 并发安全，应用级复用。
所有操作尽量用 Context。
`Rows` 必须 Close。
循环结束检查 `rows.Err()`。
事务里必须使用 `tx`，不能混用 `db`。
事务边界通常放 service 层。
复杂 SQL 和核心链路不要无脑 ORM。

---

## 【文本块 61】事务与 MVCC 速记

ACID：

* 原子性：undo log
* 持久性：redo log
* 隔离性：MVCC + 锁
* 一致性：最终目标

隔离级别：

* RU：脏读，基本不用
* RC：解决脏读，会不可重复读
* RR：解决不可重复读，MySQL 默认
* Serializable：串行化，性能差

MVCC：

* 隐藏事务 ID
* undo log 版本链
* Read View 判断可见性

快照读：普通 SELECT。
当前读：SELECT FOR UPDATE、UPDATE、DELETE。

---

## 【文本块 62】日志速记

redo log：InnoDB 物理日志，崩溃恢复，保证持久性。
undo log：旧版本，事务回滚，MVCC。
binlog：Server 层逻辑日志，主从复制和恢复。

两阶段提交：

1. redo log prepare
2. 写 binlog
3. redo log commit

目的是保证 redo log 和 binlog 一致。

---

## 【文本块 63】索引速记

InnoDB 索引是 B+ 树。
B+ 树多叉、树低、IO 少，叶子节点链表适合范围查询。
聚簇索引叶子节点存整行。
二级索引叶子节点存索引列和主键。
二级索引查整行可能回表。
覆盖索引可以避免回表。
联合索引遵循最左前缀。
范围查询后面的列通常不能继续用于有序定位。
索引失效本质是无法利用索引有序性或成本太高。

---

## 【文本块 64】锁速记

InnoDB 行锁基于索引记录。
SQL 不走索引会扩大锁范围。
S 锁共享读，X 锁排他写。
间隙锁锁范围间隙，防止插入。
Next-Key Lock = 记录锁 + 间隙锁。
RR 下通过 MVCC + Next-Key Lock 解决幻读。
死锁本质是循环等待。
避免死锁：固定访问顺序、缩短事务、走索引、小事务、必要时有限重试。

---

## 【文本块 65】主从复制速记

主从复制基于 binlog。
主库写 binlog。
从库 IO 线程拉 binlog 写 relay log。
从库 SQL 线程重放 relay log。
默认异步复制，所以会主从延迟。
主从延迟会导致写后读旧数据。
解决：强一致读主、写后短时间读主、监控延迟、延迟高摘除从库。

---

# 二十九、本章最终面试回答模板

## 【文本块 66】MySQL Go 综合回答模板

如果面试官让我系统讲 MySQL，我会按照 Go 接入、事务 MVCC、日志、索引、锁、主从和工程排查这几条线来组织。

在 Go 项目里，我通常通过 `database/sql` 加 MySQL 驱动接入数据库。这里要注意 `sql.DB` 不是单个连接，而是连接池，应该在应用启动时初始化并复用。生产里必须配置 `MaxOpenConns`、`MaxIdleConns`、`ConnMaxLifetime`、`ConnMaxIdleTime`，并且所有查询都使用 `QueryContext`、`ExecContext`、`BeginTx` 这类带 context 的 API，避免慢 SQL 或网络抖动拖死服务。查询多行时 `Rows` 必须 Close，事务中必须使用 `tx`，不能混用外层 `db`。

事务方面，MySQL InnoDB 通过 undo log 保证原子性，通过 redo log 保证持久性，通过 MVCC 和锁保证隔离性。隔离级别有 RU、RC、RR、Serializable。RC 每次快照读都会生成新的 Read View，所以能看到其他事务新提交的数据；RR 第一次快照读生成 Read View，后续复用，所以同一事务内读到的数据一致。普通 SELECT 是快照读，通过 MVCC 读取历史版本；SELECT FOR UPDATE、UPDATE、DELETE 是当前读，要读取最新数据并加锁。

日志方面，redo log 是 InnoDB 的物理日志，用于崩溃恢复；undo log 记录旧版本，用于事务回滚和 MVCC；binlog 是 Server 层逻辑日志，用于主从复制和数据恢复。事务提交时需要两阶段提交，先写 redo log prepare，再写 binlog，最后 redo log commit，目的是保证 InnoDB 崩溃恢复和 MySQL 主从复制之间的一致性。

索引方面，InnoDB 使用 B+ 树索引，因为 B+ 树是多叉树，树高低，能减少磁盘 IO，并且叶子节点有序链表适合范围查询。主键索引是聚簇索引，叶子节点存整行数据；二级索引叶子节点存索引列和主键，如果查询字段不在二级索引里，就要回表。覆盖索引可以避免回表。联合索引遵循最左前缀原则，设计时不能只看区分度，还要结合查询频率、等值条件、范围条件、排序和覆盖索引需求。

锁方面，InnoDB 的行锁本质上是锁索引记录。如果 SQL 没走索引，锁范围可能扩大。常见锁有表锁、行锁、间隙锁、Next-Key Lock，还有共享锁和排他锁。Next-Key Lock 是记录锁加间隙锁，用于防止当前读场景下的幻读。死锁的本质是循环等待，工程上要通过固定资源访问顺序、缩短事务、保证 SQL 走索引、拆分大事务来减少死锁；必要时可以对死锁做有限重试，但前提是业务幂等。

主从复制方面，MySQL 通过 binlog 实现复制。主库写 binlog，从库 IO 线程拉取 binlog 写入 relay log，从库 SQL 线程重放 relay log。默认异步复制会有主从延迟，带来写后读旧数据的问题。在 Go 服务里做读写分离时，普通读可以走从库，但强一致读、写后立即读、事务内读都应该走主库，同时要监控从库延迟，延迟过高时摘除从库或降级读主。

工程实践上，我会根据场景选择 GORM、sqlx 或手写 SQL。普通 CRUD 可以用 GORM 提升效率，但复杂 SQL、性能敏感路径和核心交易链路更倾向于 sqlx 或 database/sql。无论用什么方式，都要关注真实 SQL、执行计划、慢查询、连接池指标、锁等待和主从延迟。SQL 优化不是一上来加索引，而是先定位慢 SQL，再通过 EXPLAIN 看 type、key、rows、Extra，分析是否全表扫描、是否回表、是否 filesort、是否 temporary，最后结合业务访问模式调整 SQL 和索引。

一句话总结：

MySQL 在 Go 后端里的核心不是会写 CRUD，而是要理解 InnoDB 如何用 MVCC 解决读写并发，用锁解决写写冲突，用 redo/undo/binlog 保证事务和复制，用 B+ 树索引降低 IO；同时在 Go 工程里通过连接池、context、事务边界、慢查询监控、读写分离和幂等约束，把这些数据库能力稳定地落到生产系统中。
